# Provisional UiO-66 bare-structure preparation
Use Internet + two T4 GPUs and the HF_TOKEN secret. Upload UIO66_Phase3_bundle.zip
as a private Kaggle Dataset. Results are model-relaxed bare structures, not
adsorption labels, fitted h/J, DAC performance, or chemistry validation.
Each output path must be fresh. Preserve old runs and their failure records.


In [ ]:
import os, sys, subprocess, json, hashlib, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fairchem-core==2.23.0', 'ase==3.26.0'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)


In [ ]:
matches = list(Path('/kaggle/input').rglob('UIO66_Phase3_bundle.zip'))
assert len(matches) == 1, 'Attach exactly one UiO-66 code bundle'
UIO_ROOT = Path('/kaggle/working/uio66_code')
UIO_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(matches[0]) as archive:
    archive.extractall(UIO_ROOT)
manifest = json.loads((UIO_ROOT/'bundle_manifest.json').read_text())
for name, digest in manifest['sha256'].items():
    assert hashlib.sha256((UIO_ROOT/name).read_bytes()).hexdigest() == digest, name


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
uma_revision = HfApi().model_info('facebook/UMA', token=os.environ['HF_TOKEN']).sha
filename = 'checkpoints/uma-s-1p2p1.pt'
assert filename in HfApi().list_repo_files('facebook/UMA', revision=uma_revision, token=os.environ['HF_TOKEN'])
uma_checkpoint = hf_hub_download('facebook/UMA', filename, revision=uma_revision, token=os.environ['HF_TOKEN'])
print('Pinned UMA revision:', uma_revision)
uio_env = {**os.environ, 'PYTHONPATH': str(UIO_ROOT)}
base_command = [sys.executable, str(UIO_ROOT/'kaggle/run_uio66_relaxation.py'),
    '--structures', str(UIO_ROOT/'uio66_structures'), '--checkpoint', uma_checkpoint,
    '--checkpoint-revision', uma_revision]


In [ ]:
subprocess.run(base_command + ['--output', '/kaggle/working/uio66_pilot/relaxation.json',
    '--ids', 'uio66_000000', 'uio66_100000', 'uio66_101010', 'uio66_111111'], check=True, env=uio_env)


## Full run after pilot
The following cell evaluates all 64 structures across both GPUs. It preserves
the separate pilot and saves each candidate immediately. Force convergence
does not establish valid bonding, thermodynamic stability or adsorption quality.


In [ ]:
pilot = json.loads(Path('/kaggle/working/uio66_pilot/relaxation.json').read_text())
assert len(pilot['results']) == 4 and all(r['status'] == 'converged' and
    r['severe_contacts_below_0_7_angstrom'] == 0 for r in pilot['results'])
ids = sorted(r['id'] for r in json.loads((UIO_ROOT/'uio66_structures/manifest.json').read_text())['configurations'])
processes = [subprocess.Popen(base_command + ['--output',
    f'/kaggle/working/uio66_full/gpu{gpu}/relaxation.json', '--ids'] + ids[gpu::2],
    env={**uio_env, 'CUDA_VISIBLE_DEVICES': str(gpu)}) for gpu in range(2)]
codes = [process.wait() for process in processes]
assert codes == [0, 0], codes


In [ ]:
subprocess.run([sys.executable, str(UIO_ROOT/'scripts/audit_uio66_relaxation.py'),
    '--structures', str(UIO_ROOT/'uio66_structures'), '--reports',
    '/kaggle/working/uio66_full/gpu0/relaxation.json',
    '/kaggle/working/uio66_full/gpu1/relaxation.json', '--output',
    '/kaggle/working/uio66_full/audit.json'], check=True, env=uio_env)
import shutil
shutil.make_archive('/kaggle/working/uio66_bare_results', 'zip', '/kaggle/working/uio66_full')
print('Download uio66_bare_results.zip; adsorption and chemistry gates remain open')
